In [30]:
import asyncio
from codecs import StreamReader
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from dotenv import load_dotenv
from autogen_agentchat.ui import Console
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 6


In [31]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat


planning_agent = AssistantAgent(
    name="PlanningAgent",
    description="An agent for planning tasks, this agent should be the first to engage when given a new task.",
    model_client=model_client,
    system_message="""
    You are a planning agent.
    Your job is to break down complex tasks into smaller, manageable subtasks.
    Your team members are:
        WebSearchAgent: Searches for information
        DataAnalystAgent: Performs calculations

    You only plan and delegate tasks - you do not execute them yourself.

    When assigning tasks, use this format:
    1. <agent> : <task>

    After all tasks are complete, summarize the findings and end with "TERMINATE".
    """,
)

In [32]:
from dotenv import load_dotenv

from langchain_community.utilities import GoogleSerperAPIWrapper

from autogen_ext.tools.http import HttpTool


os.environ['SERPER_API_KEY']='bead05022450578faa7498f4c90d85e534c372e0'


search_tool_wrapper = GoogleSerperAPIWrapper(type='search')

def search_web(query:str) ->str:
    """Search the web for the given query and return the results."""
    try:
        results = search_tool_wrapper.run(query)
        return results
    except Exception as e:
        print(f"Error occurred while searching the web: {e}")
        return "No results found."

In [33]:
def search_web_tool(query:str)-> str:
    # Simulate a web search
    if "2006-2007" in query:
        return """Here are the total points scored by Miami Heat players in the 2006-2007 season:
        Udonis Haslem: 844 points
        Dwayne Wade: 1397 points
        James Posey: 550 points
        ...
        """
    elif "2007-2008" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2007-2008 is 214."
    elif "2008-2009" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2008-2009 is 398."
    return "No data found."

In [34]:
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)


In [35]:
web_search_agent = AssistantAgent(
    name = 'WebSearchAgent',
    description= 'An agent for searching the web for information.',
    model_client=model_client,
    tools = [search_web_tool],
    reflect_on_tool_use=False,
    system_message='''
        You are a web search agent.
        Your only tool is search_web - use it to find the information you need.

        You make only one search call at a time.
        
        Once you have the results, you never do calculations or data analysis on them.
    ''',
)

In [36]:
def percentage_change_tool(start:float, end:float) -> float:
    # Calculate percentage change
    if start == 0:
        return 0
    return ((end - start) / start) * 100

In [37]:
data_analyst_agent = AssistantAgent(
    name = 'DataAnalystAgent',
    description= 'An agent for performing calculations and data analysis.',
    model_client=model_client,
    tools= [percentage_change_tool],
    system_message='''
        You are a data analyst agent.
        Given the tasks you have been assigned, you should analyze the data and provide results using the tools provided (percentage_change_tool).

        If you have not seen the data, ask for it.

    ''',
)

Termination Condition


In [38]:
from autogen_agentchat.conditions import TextMentionTermination,MaxMessageTermination

text_mention_termination = TextMentionTermination('TERMINATE')
max_message_termination = MaxMessageTermination(max_messages=20)
combined_termination = text_mention_termination | max_message_termination

In [39]:
selector_prompt = '''
Select an agent to perform the task.

{roles}

current conversation history :
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure that the planning agent has assigned task before other agents start working.
Only select one agent.
'''

In [40]:
planning_agent.description


'An agent for planning tasks, this agent should be the first to engage when given a new task.'

In [41]:
selector_team = SelectorGroupChat(
    participants=[planning_agent, web_search_agent, data_analyst_agent],
    model_client=model_client,
    termination_condition=combined_termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True)

In [42]:
task = "Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?"


In [43]:
from autogen_agentchat.ui import Console

await Console(selector_team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
To address your inquiry, we need to take the following steps:

1. Identify the Miami Heat player with the highest points in the 2006-2007 season.
2. Retrieve the total rebounds for this player for the seasons 2007-2008 and 2008-2009.
3. Calculate the percentage change in total rebounds between these two seasons.

Let's begin assigning tasks:

1. WebSearchAgent: Find the Miami Heat player with the highest points in the 2006-2007 NBA season.
2. WebSearchAgent: Retrieve the total rebounds for this player for the 2007-2008 NBA season.
3. WebSearchAgent: Retrieve the total rebounds for this player for the 2008-2009 NBA season.
4. DataAnalystAgent: Calculate the percentage change in total rebounds from the 2007-2008 season to

TaskResult(messages=[TextMessage(id='573d8539-751c-410d-8b4c-1c985499a63d', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 9, 19, 22, 7, 15, 487372, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(id='be0e5e48-1ddf-4a9d-93a3-5922ba2ec72a', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=161, completion_tokens=198), metadata={}, created_at=datetime.datetime(2025, 9, 19, 22, 7, 20, 45077, tzinfo=datetime.timezone.utc), content="To address your inquiry, we need to take the following steps:\n\n1. Identify the Miami Heat player with the highest points in the 2006-2007 season.\n2. Retrieve the total rebounds for this player for the seasons 2007-2008 and 2008-2009.\n3. Calculate the percentage change in total rebounds between these two sea

In [44]:
# With real web search


from autogen_agentchat.ui import Console

await Console(selector_team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
To address your inquiry, we need to take the following steps:

1. Confirm the Miami Heat player with the highest points in the 2006-2007 season.
2. Retrieve the total rebounds for this player for the seasons 2007-2008 and 2008-2009.
3. Calculate the percentage change in total rebounds between these two seasons.

Here are the steps to complete these tasks based on the gathered data:

1. The Miami Heat player with the highest points in the 2006-2007 season was previously identified as Dwayne Wade, scoring 1397 points.
2. Dwayne Wade's total rebounds for the 2007-2008 season were 214.
3. Dwayne Wade's total rebounds for the 2008-2009 season were 398.
4. Dwayne Wade's total rebounds increased from 214 in the 2007-2008 seaso

TaskResult(messages=[TextMessage(id='588e27e3-fd5c-427e-a818-731ba8371dc5', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 9, 19, 22, 7, 29, 568603, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(id='dd4f0ffe-406d-4b0c-b529-c3d27c411271', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=676, completion_tokens=221), metadata={}, created_at=datetime.datetime(2025, 9, 19, 22, 7, 35, 463057, tzinfo=datetime.timezone.utc), content="To address your inquiry, we need to take the following steps:\n\n1. Confirm the Miami Heat player with the highest points in the 2006-2007 season.\n2. Retrieve the total rebounds for this player for the seasons 2007-2008 and 2008-2009.\n3. Calculate the percentage change in total rebounds between these two sea

In [45]:
state = await selector_team.save_state()


In [46]:
state

{'type': 'TeamState',
 'version': '1.0.0',
 'agent_states': {'PlanningAgent': {'type': 'ChatAgentContainerState',
   'version': '1.0.0',
   'agent_state': {'type': 'AssistantAgentState',
    'version': '1.0.0',
    'llm_context': {'messages': [{'content': 'Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?',
       'source': 'user',
       'type': 'UserMessage'},
      {'content': "To address your inquiry, we need to take the following steps:\n\n1. Identify the Miami Heat player with the highest points in the 2006-2007 season.\n2. Retrieve the total rebounds for this player for the seasons 2007-2008 and 2008-2009.\n3. Calculate the percentage change in total rebounds between these two seasons.\n\nLet's begin assigning tasks:\n\n1. WebSearchAgent: Find the Miami Heat player with the highest points in the 2006-2007 NBA season.\n2. WebSearchAgent: Retrieve the t

In [47]:
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from typing import Sequence

def my_selector_fun(messages: Sequence[BaseAgentEvent | BaseChatMessage]):

    if messages[-1].source == web_search_agent.name:
        return data_analyst_agent.name
    return None


selector_team = SelectorGroupChat(
    participants=[planning_agent, web_search_agent, data_analyst_agent],
    model_client=model_client,
    termination_condition=combined_termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True,
    selector_func=my_selector_fun)

In [48]:
task = "Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?"

from autogen_agentchat.ui import Console

await Console(selector_team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
To answer your question, we will follow these steps:

1. Identify the Miami Heat player with the highest points in the 2006-2007 season.
2. Retrieve the total rebounds for this player for the 2007-2008 season.
3. Retrieve the total rebounds for this player for the 2008-2009 season.
4. Calculate the percentage change in total rebounds between the two seasons.

Here are the results based on prior findings:

1. The Miami Heat player with the highest points in the 2006-2007 season was Dwayne Wade, who scored 1397 points.
2. Dwayne Wade's total rebounds in the 2007-2008 season were 214.
3. Dwayne Wade's total rebounds in the 2008-2009 season were 398.
4. The percentage increase in Dwayne Wade's total rebounds from the 2007-2

TaskResult(messages=[TextMessage(id='f017ae99-c9a4-41c3-9d0c-7d2d606ce353', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 9, 19, 22, 7, 35, 526685, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(id='c7cf4ff4-5efd-446b-b3b8-aa7bf5c5e3fd', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=953, completion_tokens=226), metadata={}, created_at=datetime.datetime(2025, 9, 19, 22, 7, 39, 757499, tzinfo=datetime.timezone.utc), content="To answer your question, we will follow these steps:\n\n1. Identify the Miami Heat player with the highest points in the 2006-2007 season.\n2. Retrieve the total rebounds for this player for the 2007-2008 season.\n3. Retrieve the total rebounds for this player for the 2008-2009 season.\n4. Calculate the percen

In [49]:

await selector_team.reset()


In [50]:
task = "Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?"

from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from typing import Sequence

def my_selector_fun(messages: Sequence[BaseAgentEvent | BaseChatMessage]):

    if messages[-1].source != web_search_agent.name:
        return data_analyst_agent.name
    return None


selector_team = SelectorGroupChat(
    participants=[planning_agent, web_search_agent, data_analyst_agent],
    model_client=model_client,
    termination_condition=combined_termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=False,
    selector_func=my_selector_fun)

from autogen_agentchat.ui import Console

await Console(selector_team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (DataAnalystAgent) ----------
Please provide the data for the Miami Heat player's points in the 2006-2007 season and their total rebounds for the 2007-2008 and 2008-2009 seasons, so I can analyze it for you.
---------- TextMessage (DataAnalystAgent) ----------
Apologies for the confusion. The Miami Heat player with the highest points in the 2006-2007 season was Dwyane Wade.  Please provide his total rebounds for the 2007-2008 and 2008-2009 seasons, so I can calculate the percentage change.
---------- TextMessage (DataAnalystAgent) ----------
Dwyane Wade was the Miami Heat player with the highest points per game in the 2006-2007 NBA season. Could you please provide his total rebound data for the 2007-2008 and 2008-2009 seasons to calculate the per

TaskResult(messages=[TextMessage(id='afc64189-a27f-4a44-b507-ce375cba15f2', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 9, 19, 22, 7, 39, 789522, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(id='34af12f1-d46c-4831-9b0f-ec15a6c2d525', source='DataAnalystAgent', models_usage=RequestUsage(prompt_tokens=150, completion_tokens=49), metadata={}, created_at=datetime.datetime(2025, 9, 19, 22, 7, 40, 790344, tzinfo=datetime.timezone.utc), content="Please provide the data for the Miami Heat player's points in the 2006-2007 season and their total rebounds for the 2007-2008 and 2008-2009 seasons, so I can analyze it for you.", type='TextMessage'), TextMessage(id='110a9274-2c91-4df5-89e2-eb417b7176a0', source='DataAnalystAgent', models_usage=RequestUsage(

In [51]:
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from typing import Sequence, List
def candidate_func(messages: Sequence[BaseAgentEvent | BaseChatMessage]) -> List[str]:
    # keep planning_agent first one to plan out the tasks
    if messages[-1].source == "user":
        return [planning_agent.name]

    # if previous agent is planning_agent and if it explicitely asks for web_search_agent
    # or data_analyst_agent or both (in-case of re-planning or re-assignment of tasks)
    # then return those specific agents
    last_message = messages[-1]
    if last_message.source == planning_agent.name:
        participants = []
        if web_search_agent.name in last_message.to_text():
            participants.append(web_search_agent.name)
        if data_analyst_agent.name in last_message.to_text():
            participants.append(data_analyst_agent.name)
        if participants:
            return participants  # SelectorGroupChat will select from the remaining two agents.

    # we can assume that the task is finished once the web_search_agent
    # and data_analyst_agent have took their turns, thus we send
    # in planning_agent to terminate the chat
    previous_set_of_agents = set(message.source for message in messages)
    if web_search_agent.name in previous_set_of_agents and data_analyst_agent.name in previous_set_of_agents:
        return [planning_agent.name]

    # if no-conditions are met then return all the agents
    return [planning_agent.name, web_search_agent.name, data_analyst_agent.name]

In [52]:
from autogen_agentchat.agents import UserProxyAgent

user_proxy_agent = UserProxyAgent("UserProxyAgent", description="A proxy for the user to approve or disapprove tasks.")

text_mention_termination = TextMentionTermination("TERMINATE")
max_messages_termination = MaxMessageTermination(max_messages=10)
termination = text_mention_termination | max_messages_termination


def selector_func_with_user_proxy(messages: Sequence[BaseAgentEvent | BaseChatMessage]) -> str | None:
    if messages[-1].source != planning_agent.name and messages[-1].source != user_proxy_agent.name:
        # Planning agent should be the first to engage when given a new task, or check progress.
        return planning_agent.name
    

    if messages[-1].source == planning_agent.name:
        if messages[-2].source == user_proxy_agent.name and "APPROVE" in messages[-1].content.upper():  # type: ignore
            # User has approved the plan, proceed to the next agent.
            return None
        # Use the user proxy agent to get the user's approval to proceed.
        return user_proxy_agent.name
    

    if messages[-1].source == user_proxy_agent.name:
        # If the user does not approve, return to the planning agent.
        if "APPROVE" not in messages[-1].content.upper():  # type: ignore
            return planning_agent.name
        

    return None

In [53]:
# Reset the previous agents and run the chat again with the user proxy agent and selector function.
# await sele.reset()
team = SelectorGroupChat(
    [planning_agent, web_search_agent, data_analyst_agent, user_proxy_agent],
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    selector_func=selector_func_with_user_proxy,
    allow_repeated_speaker=True,
)

await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
To answer your query, we need to follow these steps:

1. Identify the Miami Heat player with the highest points in the 2006-2007 season.
2. Determine this player's total rebounds in the 2007-2008 and 2008-2009 seasons.
3. Calculate the percentage change in the player's total rebounds between these seasons.

Let's begin:

1. **WebSearchAgent**: Search for the Miami Heat player with the highest points in the 2006-2007 NBA season.
   
Once the player's name is identified, the next tasks will be:

2. **WebSearchAgent**: Find the total rebounds for the identified player during the 2007-2008 season.
3. **WebSearchAgent**: Find the total rebounds for the identified player during the 2008-2009 season.
4. **DataAnalystAgent**: C

TaskResult(messages=[TextMessage(id='0a7a5656-eadd-49b5-b8b8-9948ac2e28a7', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 9, 19, 22, 8, 4, 109026, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(id='4bf49d8a-d038-4b60-989f-e5fbea3c4c78', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=161, completion_tokens=203), metadata={}, created_at=datetime.datetime(2025, 9, 19, 22, 8, 7, 291809, tzinfo=datetime.timezone.utc), content="To answer your query, we need to follow these steps:\n\n1. Identify the Miami Heat player with the highest points in the 2006-2007 season.\n2. Determine this player's total rebounds in the 2007-2008 and 2008-2009 seasons.\n3. Calculate the percentage change in the player's total rebounds between these seasons.\n